In [0]:
"""
DDL Generator — Databricks & Snowflake
=======================================
Reads SHOW CREATE TABLE output from Databricks and generates
compatible DDL for both Databricks and Snowflake.

Usage (in Databricks notebook):
    %run ./ddl_generator
    generator = DDLGenerator(spark, catalog="wh_gb_test", schema="gb_test")
    generator.export_all(output_dir="/Workspace/Users/selvakumar.ravindran@gmail.com/Databricks/WH_DATA/DDL")

Usage (standalone with raw DDL string):
    generator = DDLGenerator()
    generator.convert_ddl(raw_ddl, table_name="order")
"""

import re
import os


# ─────────────────────────────────────────────
# Type mapping: Databricks → Snowflake
# ─────────────────────────────────────────────
TYPE_MAP_TO_SNOWFLAKE = {
    "BIGINT":    "NUMBER(19,0)",
    "INT":       "NUMBER(10,0)",
    "FLOAT":     "FLOAT",
    "DOUBLE":    "FLOAT",
    "BOOLEAN":   "BOOLEAN",
    "TIMESTAMP": "TIMESTAMP_NTZ",
    "DATE":      "DATE",
    "STRING":    "VARCHAR(16777216)",
}

# Reserved words that must be quoted in both platforms
RESERVED_WORDS = {"order", "group", "select", "table", "from", "where", "return"}


# ─────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────

def _quote_databricks(name: str) -> str:
    """Wrap identifier in backticks if reserved or contains spaces."""
    if name.lower() in RESERVED_WORDS or " " in name:
        return f"`{name}`"
    return name


def _quote_snowflake(name: str) -> str:
    """Wrap identifier in double quotes if reserved or mixed-case."""
    if name.lower() in RESERVED_WORDS or " " in name:
        return f'"{name}"'
    return name


def _map_type_to_snowflake(col_type: str) -> str:
    """Convert Databricks type to Snowflake equivalent."""
    upper = col_type.strip().upper()
    # VARCHAR(n) — keep as-is
    if upper.startswith("VARCHAR"):
        return col_type.strip()
    # DECIMAL(p,s) — keep as-is
    if upper.startswith("DECIMAL") or upper.startswith("NUMERIC"):
        return col_type.strip()
    return TYPE_MAP_TO_SNOWFLAKE.get(upper, col_type.strip())


def _strip_databricks_extras(ddl: str) -> str:
    """Remove Databricks-specific clauses from raw SHOW CREATE TABLE output."""
    # Remove COLLATE clauses on columns
    ddl = re.sub(r"\s+COLLATE\s+\w+", "", ddl, flags=re.IGNORECASE)
    # Remove DEFAULT COLLATION block
    ddl = re.sub(r"\s*DEFAULT\s+COLLATION\s+\w+", "", ddl, flags=re.IGNORECASE)
    # Remove USING DELTA
    ddl = re.sub(r"\s*USING\s+DELTA", "", ddl, flags=re.IGNORECASE)
    # Remove TBLPROPERTIES block (multiline)
    ddl = re.sub(r"\s*TBLPROPERTIES\s*\(.*?\)", "", ddl, flags=re.IGNORECASE | re.DOTALL)
    # Remove NOT ENFORCED
    ddl = re.sub(r"\s+NOT\s+ENFORCED", "", ddl, flags=re.IGNORECASE)
    # Remove GENERATED ALWAYS AS IDENTITY / GENERATED BY DEFAULT AS IDENTITY
    ddl = re.sub(r"\s*GENERATED\s+(ALWAYS|BY\s+DEFAULT)\s+AS\s+IDENTITY", "", ddl, flags=re.IGNORECASE)
    # Remove AUTO_INCREMENT
    ddl = re.sub(r"\s*AUTO_INCREMENT", "", ddl, flags=re.IGNORECASE)
    return ddl.strip()


def _extract_table_name(ddl: str) -> str:
    """Extract unqualified table name from CREATE TABLE statement."""
    match = re.search(r"CREATE\s+(?:OR\s+REPLACE\s+)?TABLE\s+[\w`.\"]+\.?([\w`\"]+)\s*\(", ddl, re.IGNORECASE)
    if match:
        return match.group(1).strip("`\"")
    return "unknown_table"


# ─────────────────────────────────────────────
# Core converter
# ─────────────────────────────────────────────

class DDLConverter:
    """Converts a raw Databricks DDL string to Databricks-clean and Snowflake DDL."""

    def __init__(self, raw_ddl: str, table_name: str = None, databricks_catalog_schema: str = None, snowflake_schema: str = None):
        """
        Args:
            raw_ddl:                     Raw output from SHOW CREATE TABLE
            table_name:                  Override table name (optional)
            databricks_catalog_schema:   e.g. "wh_gb_test.gb_test" for Databricks output
            snowflake_schema:            e.g. "gb_test" for Snowflake output
        """
        self.raw_ddl                  = raw_ddl
        self.table_name               = table_name or _extract_table_name(raw_ddl)
        self.databricks_catalog_schema = databricks_catalog_schema or ""
        self.snowflake_schema         = snowflake_schema or ""

    def _base_ddl(self) -> str:
        """Strip all platform-specific extras, leaving clean ANSI-ish DDL."""
        return _strip_databricks_extras(self.raw_ddl)

    def _replace_backticks_with_double_quotes(self, ddl: str) -> str:
        """Replace all backtick-quoted identifiers with double-quote equivalents."""
        return re.sub(r"`([^`]+)`", r'"\1"', ddl)

    def _fix_table_ref(self, ddl: str, quote_fn, schema: str) -> str:
        """
        Replace fully-qualified table references with correct schema + quoting.
        Handles: catalog.schema.table  →  schema.table
        """
        def replacer(m):
            tname = m.group(3).strip("`\"")
            qtname = quote_fn(tname)
            if schema:
                return f"{schema}.{qtname}"
            return qtname

        # Match 3-part: catalog.schema.table (backtick or plain)
        ddl = re.sub(
            r"[`\"]?[\w]+[`\"]?\.[`\"]?[\w]+[`\"]?\.[`\"]?([\w]+)[`\"]?",
            lambda m: (
                f"{schema}.{quote_fn(m.group(1))}" if schema
                else quote_fn(m.group(1))
            ),
            ddl
        )
        return ddl

    def to_databricks(self) -> str:
        """Generate clean Databricks-compatible DDL."""
        ddl = self._base_ddl()

        # Normalize table name with catalog.schema prefix
        tname = _quote_databricks(self.table_name)
        prefix = f"{self.databricks_catalog_schema}." if self.databricks_catalog_schema else ""

        # Replace CREATE TABLE header
        ddl = re.sub(
            r"CREATE\s+(?:OR\s+REPLACE\s+)?TABLE\s+[\w`.\"]+",
            f"CREATE OR REPLACE TABLE {prefix}{tname}",
            ddl, count=1, flags=re.IGNORECASE
        )

        # Fix FK references to use catalog.schema prefix
        if self.databricks_catalog_schema:
            ddl = re.sub(
                r"REFERENCES\s+[`\"]?[\w]+[`\"]?\.[`\"]?[\w]+[`\"]?\.[`\"]?([\w]+)[`\"]?",
                lambda m: f"REFERENCES {self.databricks_catalog_schema}.{_quote_databricks(m.group(1))}",
                ddl, flags=re.IGNORECASE
            )

        # Map column types
        ddl = self._map_types(ddl, target="databricks")

        return (
            ddl.rstrip().rstrip(";")
            + "\nUSING DELTA;"
        )

    def to_snowflake(self) -> str:
        """Generate Snowflake-compatible DDL."""
        ddl = self._base_ddl()

        # Replace backticks with double quotes
        ddl = self._replace_backticks_with_double_quotes(ddl)

        # Normalize table name
        tname = _quote_snowflake(self.table_name)
        prefix = f"{self.snowflake_schema}." if self.snowflake_schema else ""

        # Replace CREATE TABLE header
        ddl = re.sub(
            r"CREATE\s+(?:OR\s+REPLACE\s+)?TABLE\s+[\w`.\"]+",
            f"CREATE OR REPLACE TABLE {prefix}{tname}",
            ddl, count=1, flags=re.IGNORECASE
        )

        # Fix FK references
        if self.snowflake_schema:
            ddl = re.sub(
                r'REFERENCES\s+"?[\w]+"?\."?[\w]+"?\."?([\w]+)"?',
                lambda m: f'REFERENCES {self.snowflake_schema}.{_quote_snowflake(m.group(1))}',
                ddl, flags=re.IGNORECASE
            )

        # Map column types
        ddl = self._map_types(ddl, target="snowflake")

        return ddl.rstrip().rstrip(";") + ";"

    def _map_types(self, ddl: str, target: str) -> str:
        """Replace Databricks types with target platform equivalents."""
        if target == "snowflake":
            for db_type, sf_type in TYPE_MAP_TO_SNOWFLAKE.items():
                # Match whole word only, not inside VARCHAR etc.
                ddl = re.sub(rf"\b{db_type}\b", sf_type, ddl, flags=re.IGNORECASE)
            # TIMESTAMP → TIMESTAMP_NTZ
            ddl = re.sub(r"\bTIMESTAMP\b(?!_NTZ)", "TIMESTAMP_NTZ", ddl, flags=re.IGNORECASE)
        return ddl


# ─────────────────────────────────────────────
# Orchestrator (Databricks notebook usage)
# ─────────────────────────────────────────────

class DDLGenerator:
    """
    Reads all tables from a Databricks catalog/schema using SHOW CREATE TABLE,
    then generates DDL files for both Databricks and Snowflake.

    Requires: active SparkSession (run inside Databricks notebook)
    """

    def __init__(self, spark=None, catalog: str = "", schema: str = "",
                 snowflake_schema: str = ""):
        self.spark            = spark
        self.catalog          = catalog
        self.schema           = schema
        self.snowflake_schema = snowflake_schema or schema
        self.db_prefix        = f"{catalog}.{schema}" if catalog and schema else schema

    def get_tables(self) -> list:
        """Return list of table names in the schema."""
        rows = self.spark.sql(f"SHOW TABLES IN {self.db_prefix}").collect()
        return [r["tableName"] for r in rows]

    def get_raw_ddl(self, table_name: str) -> str:
        """Fetch raw DDL from Databricks using SHOW CREATE TABLE."""
        rows = self.spark.sql(f"SHOW CREATE TABLE {self.db_prefix}.{table_name}").collect()
        return rows[0][0]

    def export_all(self, output_dir: str = "/Workspace/Users/selvakumar.ravindran@gmail.com/Databricks/WH_DATA/DDL"):
        """
        Export DDL for all tables to:
            output_dir/databricks/<table>.sql
            output_dir/snowflake/<table>.sql
        """
        db_dir = os.path.join(output_dir, "databricks")
        sf_dir = os.path.join(output_dir, "snowflake")
        os.makedirs(db_dir, exist_ok=True)
        os.makedirs(sf_dir, exist_ok=True)

        tables = self.get_tables()
        print(f"Found {len(tables)} tables: {tables}\n")

        for table in tables:
            raw_ddl = self.get_raw_ddl(table)
            converter = DDLConverter(
                raw_ddl=raw_ddl,
                table_name=table,
                databricks_catalog_schema=self.db_prefix,
                snowflake_schema=self.snowflake_schema
            )

            # Write Databricks DDL
            db_ddl = converter.to_databricks()
            db_file = os.path.join(db_dir, f"{table}.sql")
            with open(db_file, "w") as f:
                f.write(db_ddl)

            # Write Snowflake DDL
            sf_ddl = converter.to_snowflake()
            sf_file = os.path.join(sf_dir, f"{table}.sql")
            with open(sf_file, "w") as f:
                f.write(sf_ddl)

            print(f"✅ {table}")
            print(f"   Databricks → {db_file}")
            print(f"   Snowflake  → {sf_file}\n")

        print("Done. Push to GitHub via Repos UI or git commands.")


# ─────────────────────────────────────────────
# Standalone test (outside Databricks)
# ─────────────────────────────────────────────

if __name__ == "__main__":
    sample_ddl = """
    CREATE TABLE wh_gb_test.gb_test.order (
      Order_ID VARCHAR(16) COLLATE UTF8_BINARY NOT NULL,
      Customer_ID VARCHAR(8) COLLATE UTF8_BINARY NOT NULL,
      Order_Priority VARCHAR(8) COLLATE UTF8_BINARY,
      Order_Date TIMESTAMP,
      Market VARCHAR(8) COLLATE UTF8_BINARY,
      CONSTRAINT `pk_order` PRIMARY KEY (`Order_ID`) NOT ENFORCED,
      CONSTRAINT `fk_customer` FOREIGN KEY (`Customer_ID`)
        REFERENCES `wh_gb_test`.`gb_test`.`customer` (`Customer_ID`) NOT ENFORCED
    )
    USING DELTA
    DEFAULT COLLATION UTF8_BINARY
    TBLPROPERTIES (
      'delta.minReaderVersion' = '3',
      'delta.minWriterVersion' = '7'
    )
    """

    converter = DDLConverter(
        raw_ddl=sample_ddl,
        table_name="order",
        databricks_catalog_schema="wh_gb_test.gb_test",
        snowflake_schema="gb_test"
    )

    print("=" * 60)
    print("DATABRICKS DDL")
    print("=" * 60)
    print(converter.to_databricks())

    print("\n" + "=" * 60)
    print("SNOWFLAKE DDL")
    print("=" * 60)
    print(converter.to_snowflake())